### Mini Project: Food Delivery Review Intelligence System

Objective:
* Build a console-based NLP application for a food delivery platform that lets staff add customer
reviews, classifies each review by sentiment and complaint category, and generates a summary
report — combining text preprocessing, TF-IDF vectorisation, and Naive Bayes or Logistic
Regression classification into a single working program.

Your project must:
* The program must be menu-driven with at least 3 options: (1) Add a new review, (2) Classify a
review and display the predicted sentiment and issue category, (3) View a summary report.
* Every review must pass through the full preprocessing pipeline (lowercase, punctuation
removal, stopword removal, stemming or lemmatization) before being stored or classified.
* The classifier must output both a sentiment label (Positive / Negative) and an issue category
(Delivery / Food Quality / App / General) for every submitted review.
* The summary report must display: total reviews added, count per sentiment label, count per
issue category, and the 5 most frequent content words across all stored reviews.
* The program must handle edge cases gracefully — if fewer than the minimum required
training samples exist, it must display a clear message instead of crashing when classification
is attempted.

In [ ]:
import string
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def preprocess_review(review):
    review = review.lower()
    for p in string.punctuation:
        review = review.replace(p, "")
    words = review.split()
    stop_words = stopwords.words('english')
    clean_words = []
    for w in words:
        if w not in stop_words:
            clean_words.append(w)
    stemmer = PorterStemmer()
    final_words = []
    for w in clean_words:
        final_words.append(stemmer.stem(w))
    return " ".join(final_words)


class ReviewIntelligenceSystem:
    def __init__(self):
        self.reviews = []
        self.labels_sentiment = []
        self.labels_issue = []
        self.is_trained = False
        
        self.pipe_sentiment = Pipeline([
            ('tfidf', TfidfVectorizer(preprocessor=preprocess_review)),
            ('lr', LogisticRegression())
        ])
        
        self.pipe_issue = Pipeline([
            ('tfidf', TfidfVectorizer(preprocessor=preprocess_review)),
            ('nb', MultinomialNB())
        ])
        
    def train_models(self):
        if len(self.reviews) < 5:
            print("\n[Error] Not enough data to train models. Need at least 5 reviews.")
            return False
            
        try:
            self.pipe_sentiment.fit(self.reviews, self.labels_sentiment)
            self.pipe_issue.fit(self.reviews, self.labels_issue)
            self.is_trained = True
            return True
        except Exception as e:
            print(f"\n[Error] Training failed: {e}\nEnsure you have a mix of different labels.")
            return False
            
    def add_review(self):
        print("\n--- Add New Review ---")
        review_text = input("Enter review text: ")
        sentiment = input("Enter sentiment (Positive/Negative): ").strip().capitalize()
        issue = input("Enter issue category (Delivery/Food Quality/App/General): ").strip().title()
        
        self.reviews.append(review_text)
        self.labels_sentiment.append(sentiment)
        self.labels_issue.append(issue)
        print("Review added and preprocessed successfully!")
        self.is_trained = False 
        
    def classify_review(self):
        print("\n--- Classify Review ---")
        if not self.is_trained:
            if not self.train_models():
                return
                
        review_text = input("Enter a new review to classify: ")
        pred_sent = self.pipe_sentiment.predict([review_text])[0]
        pred_iss = self.pipe_issue.predict([review_text])[0]
        
        print(f"\nPredicted Sentiment: {pred_sent}")
        print(f"Predicted Issue Category: {pred_iss}")
        
    def view_summary(self):
        print("\n--- Summary Report ---")
        print(f"Total Reviews Added: {len(self.reviews)}")
        
        print("\nSentiment Counts:")
        sent_counts = pd.Series(self.labels_sentiment).value_counts()
        for k, v in sent_counts.items(): print(f"  {k}: {v}")
            
        print("\nIssue Category Counts:")
        iss_counts = pd.Series(self.labels_issue).value_counts()
        for k, v in iss_counts.items(): print(f"  {k}: {v}")
            
        if len(self.reviews) > 0:
            print("\nTop 5 Frequent Words:")
            all_processed = [preprocess_review(r) for r in self.reviews]
            all_words = ' '.join(all_processed).split()
            word_counts = pd.Series(all_words).value_counts().head(5)
            for k, v in word_counts.items(): print(f"  {k}: {v}")
                
    def run(self):
        while True:
            print("\n=== Review Intelligence System ===")
            print("1. Add a new review (Training Data)")
            print("2. Classify a review")
            print("3. View summary report")
            print("4. Exit")
            
            choice = input("Select an option: ")
            if choice == '1':
                self.add_review()
            elif choice == '2':
                self.classify_review()
            elif choice == '3':
                self.view_summary()
            elif choice == '4':
                print("Exiting...")
                break
            else:
                print("Invalid choice.")

if __name__ == "__main__":
    app = ReviewIntelligenceSystem()
    app.run()


=== Review Intelligence System ===
1. Add a new review (Training Data)
2. Classify a review
3. View summary report
4. Exit

--- Add New Review ---
Review added and preprocessed successfully!

=== Review Intelligence System ===
1. Add a new review (Training Data)
2. Classify a review
3. View summary report
4. Exit

--- Classify Review ---

[Error] Not enough data to train models. Need at least 5 reviews.

=== Review Intelligence System ===
1. Add a new review (Training Data)
2. Classify a review
3. View summary report
4. Exit

--- Summary Report ---
Total Reviews Added: 1

Sentiment Counts:
  Positive: 1

Issue Category Counts:
  Defect: 1

Top 5 Frequent Words:
  good: 1

=== Review Intelligence System ===
1. Add a new review (Training Data)
2. Classify a review
3. View summary report
4. Exit
Exiting...
